_____

<table align="left" width=100%>
    <td>
        <div style="text-align: center;">
          <img src="./images/bar.png" alt="entidades financiadoras"/>
        </div>
    </td>
    <td>
        <p style="text-align: center; font-size:24px;"><b>Introduction to Data Science</b></p>
        <p style="text-align: center; font-size:18px;"><b>Master in Electrical and Computer Engineering</b></p>
        <p style="text-align: center; font-size:14px;"><b>Pedro Cardoso (pcardoso@ualg.pt)</b></p>
    </td>
</table>

_____


**Lesson:** Outlier Detection and Treatment in Time Series

**Summary:**  This lesson builds upon previous time series introductions, focusing on identifying and handling outliers within time series data. Students will learn various outlier detection methods specific to time series, including techniques based on statistical measures (e.g., moving averages, standard deviation thresholds), visualization (e.g., box plots, scatter plots), and more advanced methods like anomaly detection algorithms (e.g., Isolation Forest adapted for time series, One-Class SVM). The lesson will cover strategies for outlier treatment, such as removal, imputation (using interpolation or other time series-specific methods), and transformation techniques to mitigate the impact of outliers. Students will gain practical experience applying these techniques to real-world datasets and understand the trade-offs associated with different outlier handling approaches.

# Outliers and Anomaly Detection in Time Series

Event detection in time series involves identifying significant changes or anomalies in the data that could be indicative of important events or patterns. There are several methods and techniques for event detection in time series, and the choice of method depends on the specific application and characteristics of the data.

In this context, **outliers** and **anomalies** are often used interchangeably, but they refer to different concepts in time series analysis:

- An **outlier** is a data point that is significantly different from the other data points in a dataset. Outliers can be caused by measurement errors, data processing errors, or legitimate extreme events. They are typically identified using statistical methods such as the Z-score, modified Z-score, or box-plot method.

- An **anomaly** is a pattern in the data that deviates significantly from the expected pattern. Anomalies can be caused by unusual events or changes in the underlying process that generates the data, and are identified using time series analysis methods such as trend analysis, seasonality analysis, or decomposition.

In other words, **an outlier is a single data point that stands out from the rest of the data, while an anomaly is a pattern that deviates from the expected behaviour**. Outliers may or may not be anomalies depending on the context and the analysis being performed.

Note that the terms "outlier" and "anomaly" are sometimes used interchangeably across different fields, each with their own definitions. In general, the goal of identifying outliers and anomalies is to detect unusual or unexpected events in the data, which may require further investigation or corrective action.

In this context, **events** are occurrences of interest to the analyst or user — planned events (such as a product launch or a public holiday) or unplanned events (such as a natural disaster or equipment failure). Events can be associated with a specific timestamp and used to explain changes or anomalies in the time series, or to inform forecasts based on similar past occurrences.

Some common approaches to anomaly/event/outlier detection in time series are:

- **Statistical methods**: Analyse the statistical properties of the time series to identify anomalies.
- **Machine learning methods**: Train models on historical data to learn normal patterns and flag deviations. Applicable to both supervised and unsupervised settings.
- **Time series decomposition methods**: Decompose a time series into trend, seasonality, and residual components; anomalies appear as deviations from the expected pattern in one or more components.

## Statistical Methods

This section presents statistical methods for anomaly/event/outlier detection. To illustrate these methods, the household electricity consumption dataset is used (`data/house_consumption_TS/house_consumption.csv`). The data is sampled every 15 minutes and is resampled to 6-hour intervals, focusing on the period from September to December 2022 to simplify the analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# %matplotlib widget # use this to allow to interact with the plots

# load data
df = pd.read_csv('./data/house_consumption_TS/house_consumption.csv')
df.date = pd.to_datetime(df.date)

# set the index to be the date and resample to 6h consumption
df.set_index('date', inplace=True)
df = df.resample('6h').sum()/4

# filter data to have only the months from september to December 2022 records
query = (df.index >= '2022-09-01') & (df.index <= '2022-12-31')
df = df[query]

# visualize the data
df.plot(
    figsize=(15, 6), 
    title='House consumption', 
    ylabel='Power (kW)', 
    xlabel='Date', 
    legend=False,
    grid=True,
    rot=45
)

plt.show()

An important property to verify before applying most statistical methods is whether the data follows a **normal distribution**, as many of these methods assume normality.

In [ ]:
df.kw.hist(bins=50)

The data is not perfectly normally distributed, but it is sufficiently close to justify the use of standard statistical detection methods. The following sections apply each method in turn.

### Z-score Method

The **Z-score method** identifies data points that lie more than a given number of standard deviations from the mean. Data points with Z-scores greater than 3 (or less than −3) are typically flagged as anomalies.

Steps to apply the Z-score method:

1. Compute the mean $\mu$ and standard deviation $\sigma$ of the time series.

2. Calculate the Z-score for each data point using:
$$Z = \frac{x - \mu}{\sigma},$$
where $x$ is the data point, $\mu$ is the mean, and $\sigma$ is the standard deviation.

3. Choose a threshold $\tau$ (commonly 3) above which a data point is considered an outlier.

4. Flag data points where $|Z| > \tau$ as potential anomalies.

The diagram below illustrates the key idea: the Z-score measures how many standard deviations a data point lies from the mean. Points within ±3σ account for 99.7% of the data under a normal distribution (blue region); points beyond that threshold are flagged as potential outliers (red regions).

![Z-score concept](images/zscore_concept.png)

The following function computes the Z-score and returns the potential outliers:

In [ ]:
def z_score(serie, threshold_value=3):
    """ Compute the Z-score
    """

    #compute mean and std dev
    _mean = serie.mean()
    _std_dev = serie.std()
    print(f'mean: {_mean}')
    print(f'std: {_std_dev}')

    # calculate Z-scores for each data point
    z_scores = (serie - _mean) / _std_dev

    # identify data points with Z-scores above threshold as potential anomalies
    query = (z_scores < -threshold_value) | (z_scores > threshold_value)
    return serie[query], _mean, _std_dev


Now, apply the function to the house consumption data:

In [ ]:
#define the threshold
threshold = 3

# call the z_score method
potential_outliers, mean, std_dev = z_score(df.kw, threshold)

print("Potential outliers:")
print(potential_outliers)

Define a method to plot the time series and the potential outliers:

In [ ]:
def plot_outliers(df, potential_outliers, h_lines_to_plot):
    """ Plot the time series and the potential outliers
    :param df: the time series data
    :param potential_outliers: the potential outliers
    :param h_lines_to_plot: the horizontal lines to plot
    """

    # plot the time series
    ax = df.plot(
        figsize=(15, 6),
        title='House consumption', 
        ylabel='Power (kW)', 
        xlabel='Date', 
        grid=True,
        rot=45)

    # plot the potential outliers
    ax.scatter(
        x=potential_outliers.index.values, 
        y=potential_outliers.values, 
        marker='X', 
        c='r',
        label='Outliers'
    )

    # plot horizontal lines given in the h_lines_to_plot, usually  mean - threshold*std_dev, mean and mean + threshold*std_dev
    for y in h_lines_to_plot:
        ax.axhline(y, color = 'r', linestyle = '--')

    # fill the area between the horizontal lines
    ax.fill_between(
        x=df.index.values,
        y1=min(h_lines_to_plot),
        y2=max(h_lines_to_plot),
        alpha=0.05, color="g"
    )

    # add the legend
    ax.legend()

    # show the plot
    plt.show()


Plot the time series with flagged outliers:

In [ ]:
plot_outliers(df=df,
              potential_outliers=potential_outliers,
              h_lines_to_plot=[mean, mean - threshold*std_dev, mean + threshold*std_dev])

The method identifies peak values as outliers. However, the sharp drop at the end of the series — which may represent a significant event — is not flagged, since it falls below the mean rather than above it.

A lower threshold can be used to capture more potential outliers:

In [ ]:
#define the threshold
threshold = 2

# call the z_score method
potential_outliers, mean, std_dev = z_score(df.kw, threshold)

print()
print("Potential outliers:")
print(potential_outliers)

# plot the results
plot_outliers(df=df,
              potential_outliers=potential_outliers,
                h_lines_to_plot=[mean, mean - threshold*std_dev, mean + threshold*std_dev])


The risk of diminishing the threshold is to increase the number of false positives, i.e., data points that are identified as outliers but are not. This is a trade-off that needs to be considered when choosing the threshold value for the Z-score method. In practice, the threshold value can be adjusted based on the specific characteristics of the data and the goals of the analysis.

A big number of false positives can be a problem, as maintenance teams, with time, may start to ignore the alerts, as they are not real. This can lead to a decrease in the effectiveness of the anomaly detection system and increase the risk of missing real anomalies.

### Grubbs' test

**Grubbs' test** identifies the most extreme data point in a univariate dataset. It calculates the G-statistic for each data point and compares it to the critical value of the G-statistic for a given significance level. If the G-statistic for a data point is greater than the critical value, the data point is considered an outlier.

This is a parametric test that assumes the data is normally distributed. The test compares the absolute difference between an observed data point and the sample mean to the standard deviation of the sample, and if this difference is found to be unusually large, the data point is considered an outlier.

Grubbs' test can be used to detect a single outlier in a dataset or multiple outliers. To perform the test, the following steps can be taken:

1. Determine the Grubbs' test **null hypothesis: _the dataset is normally distributed and contains no outliers_.**

2. Calculate the test statistic: The test statistic is the largest difference between a data point and the sample mean, divided by the sample standard deviation. This is represented by the formula:
$$G = \frac{\max(|X_i - \bar X|)}{\sigma}$$
where $X_i$ is the $i$-th data point, $\bar X$ is the sample mean, and $\sigma$ is the sample standard deviation.

3. Determine the critical value: The critical value is determined based on the significance level, degrees of freedom, and the number of data points in the dataset. The critical value is used to determine whether the test statistic is large enough to reject the null hypothesis, value determined by:
$$ G_{critical} = \frac{N-1}{\sqrt{N}} \sqrt{\frac{t^2_{\frac{\alpha}{2N}, N-2}}{N-2+t^2_{\frac{\alpha}{2N}, N-2}}}$$
with $t_{\alpha/(2N),N-2}$ denoting the critical value of the $t-$ distribution with $N-2$ degrees of freedom and a significance level of $\alpha/(2N)$.

4. Compare the test statistic to the critical value: If the test statistic is greater than the critical value, then the null hypothesis is rejected, and the data point corresponding to the test statistic is considered an outlier. I.e., **if $G > G_{critical}$, then the data is not normally distributed or contains an outlier.**

Grubbs' test is widely used in many fields, including engineering, environmental science, and finance, where it can be used to detect anomalies in data such as pollutant concentrations, sensor readings, and financial returns.

In [ ]:
import scipy.stats
import numpy as np

def grubbs_test(series, alpha):
    """
    x: vector of values
    alpha: "significance"
    """
    # calculate mean and standard deviation
    n = len(series)
    mean = np.mean(series)
    sts_dev = np.std(series)

    # calculate test statistic
    t_value = scipy.stats.t.ppf(1 - alpha / (2 * n), n - 2)

    # calculate critical value
    g_critical = (n - 1) * np.sqrt( np.square(t_value) / (n * (n - 2 + np.square(t_value))))
    print("Grubbs Critical Value:",g_critical)

    # find and return outliers
    return g_critical > abs(series - mean) / sts_dev, g_critical

In [ ]:
test_results, g_critical = grubbs_test(df.kw.values, alpha=0.05)
potential_outliers = df[~test_results]

print('Potential outliers:')
print(potential_outliers)

In [ ]:
plot_outliers(df=df,
              potential_outliers=potential_outliers,
              h_lines_to_plot=[mean, mean - g_critical*std_dev, mean + g_critical*std_dev])

Different levels of significance can be used to determine the critical value. For example, a significance level of 0.05 is commonly used, which corresponds to a 95% confidence level. This means that if the test is performed multiple times, 95% of the time the test will correctly identify the outlier. However, this also means that 5% of the time the test will incorrectly identify a data point as an outlier, even though it is not.

### Modified Z-score method

The **modified Z-score** method is a variation of the Z-score method that is **more robust to outliers**. It uses the median absolute deviation (MAD) instead of the standard deviation to calculate the Z-score. Typically, data points with modified Z-scores greater than 3.5 or less than -3.5 are considered anomalies.

The median absolute deviation (MAD) is a measure of the variability of a univariate dataset. It is a robust alternative to the standard deviation that is less sensitive to outliers in the data. MAD is calculated as the median of the absolute deviations of the data points from their median.

The MAD of a dataset can be computed using the following formula:
$$MAD = median(|x_i - median(X)|)$$
where $x_i$ is the $i$-th observation in the dataset $X$ and $median(X)$ is the median of $X$.

Then the modified Z-score for each data point is calculated as:
$$M_i = 0.6745 \frac{x_i - median(X)}{MAD}$$
where 
- $M_i$ is the modified Z-score for the $i$-th data point, 
- $x_i$ is the $i$-th observation in the dataset $X$, 
- $median(X)$ is the median of $X$,  
- $MAD$ is the median absolute deviation of $X$, and
- the constant 0.6745 in the numerator is used to ensure that the modified Z-score behaves like a standard Z-score for large samples of normal data, where a score of approximately ±3.5 suggests an outlier.

In [ ]:
from math import fabs

def modified_z_score(df, threshold=3):
    # calculate median absolute deviation
    median = df.kw.median()
    mad = (df.kw - median).apply(fabs).median()

    print(f'median: {median}')
    print(f'MAD: {mad}')

    # calculate modified Z-scores for each data point
    m_z_scores = 0.6745 * (df.kw - median) / mad

    # identify data points above threshold as potential outliers
    query = (m_z_scores < -threshold) | (m_z_scores > threshold)
    potential_outliers = df[query]

    return potential_outliers

In [ ]:
threshold = 3.5
potential_outliers = modified_z_score(df, threshold)

print()
print("Potential outliers:")
potential_outliers

In [ ]:
median = df.kw.median()
mad = (df.kw - median).apply(fabs).median()

plot_outliers(df=df,
              potential_outliers=potential_outliers,
              h_lines_to_plot=[median, median - threshold*mad/0.6745, median + threshold*mad/0.6745])

### Percentile method
The percentile method identifies data points that fall outside a certain percentile range. For example, data points that fall outside the 1st and 99th percentiles may be considered anomalies.


In [ ]:
p01 = df.kw.quantile(0.01)
p99 = df.kw.quantile(0.99)
query = (df.kw < p01) | (df.kw > p99)

potential_outliers = df[query]

print()
print("Potential anomalies:")
potential_outliers

In [ ]:
# plot it
plot_outliers(df=df,
              potential_outliers=potential_outliers,
              h_lines_to_plot=[p01, p99])

### Box plot method

The box plot method uses the interquartile range (IQR) to identify outliers. The box plot consists of a box and whiskers that represent the distribution of the data. The box represents the middle 50% of the data, with the bottom of the box indicating the 25th percentile and the top indicating the 75th percentile. The whiskers extend from the box to the minimum and maximum values, but they do not include any points that are identified as outliers. 

To identify outliers, the interquartile range (IQR) is used. The interquartile range (IQR) is the distance between the 25th (Q1) and 75th (Q3) percentiles of the data. You can compute the IQR by subtracting the 25th percentile from the 75th percentile, i.e.,

$$IQR = Q3 - Q1$$

Then to identify outliers using a box plot, you need to look for any points that fall outside the whiskers. The whiskers extend to 1.5 times the IQR below the 25th percentile and above the 75th percentile. Any point that falls outside the whiskers is considered an outlier.

It's important to note that not all data sets have outliers. If your data set has no outliers, then the whiskers on the box plot will extend to the minimum and maximum values of the data.

In [ ]:
q1 = df.kw.quantile(.25)
q3 = df.kw.quantile(.75)

iqr = q3 - q1

print('q1 = ', q1)
print('q3 = ', q3)
print('iqr = ', iqr)

query = (df.kw < q1 - 1.5*iqr) | (df.kw > q3 + 1.5*iqr)
potential_outliers = df[query]

print()
print("Potential outliers:")
print(potential_outliers)

In [ ]:
plot_outliers(df=df,
              potential_outliers=potential_outliers,
              h_lines_to_plot=[q1, q3, q1 - 1.5*iqr, q3 + 1.5*iqr])

Plot the time series and box plot side by side:

In [ ]:
fig, ax = plt.subplots(1, 2, sharey=True)

# plot the boxplot
df.kw.plot(kind='box', ax=ax[0])

# timeseries plot
df.plot(figsize=(15, 6), ax=ax[1])
ax[1].scatter(x=potential_outliers.index.values, y=potential_outliers.kw.values, marker='X', c='r')

# plot horizontal lines 
for y in [q1, q3, q1 - 1.5*iqr, q3 + 1.5*iqr]:
    ax[1].axhline(y, color = 'r', linestyle = '--')

ax[1].fill_between(x=df.index.values,
                   y1=q1 - 1.5*iqr,
                   y2=q3 + 1.5*iqr,
                   alpha=0.1, color="g")

### Moving Average Method

The moving average method detects anomalies by comparing each data point to a smoothed version of the series. Deviations that exceed a threshold (typically a multiple of the local standard deviation) are flagged as anomalies.

- **Step 1 — Choose a window size**: The window size controls how many data points are used in each average. A larger window smooths out more noise but may miss abrupt changes; a smaller window is more sensitive but noisier.

- **Step 2 — Calculate the moving average**: Compute the rolling mean over each window.

- **Step 3 — Calculate the deviation**: Subtract the moving average from each observed value:
$$\delta_t = x_t - \overline{x}_t$$

- **Step 4 — Set a threshold**: Compute the standard deviation of the deviations and flag points where $|\delta_t|$ exceeds a multiple (e.g. 3×) of that standard deviation.

- **Step 5 — Visualise the results**: Plot the original series, the moving average, and the flagged anomalies together.

In [ ]:
# define the windows size
window_size = 12

# make of the copy of the data
df_mam = df.copy()

# Calculate the moving average
df_mam['moving_average'] = df_mam.kw.rolling(window_size, min_periods=1).mean()

# Calculate the deviation from the moving average
df_mam['deviation'] = df_mam.kw - df_mam.moving_average

# Calculate the standard deviation of the deviations
std_dev = df_mam.deviation.std()

# Identify anomalies
threshold = 3
query = (df_mam.deviation < -threshold*std_dev) | (df_mam.deviation > threshold*std_dev)
potential_anomalies = df_mam.kw[query]

print("Potential anomalies:")
print(potential_anomalies)

In [ ]:
# Visualize the results
ax=df_mam[['kw', 'moving_average']]\
    .plot(
        figsize=(15, 6),
        title='House consumption', 
        ylabel='Power (kW)', 
        xlabel='Date', 
        legend=False,
        grid=True,
        rot=45
)

# plot the tolerance bandhe 
ax.scatter(
    x=potential_anomalies.index.values, 
    y=potential_anomalies.values, 
    color='red', 
    label='Anomalies',
    marker='X'
)

ax.fill_between(
    x=df_mam.index.values,
    y1=(df_mam.moving_average.values-threshold*std_dev),
    y2=(df_mam.moving_average.values+threshold*std_dev),
    alpha=0.1, 
    color="g",
    label='Tolerance band'

)

plt.legend()

### Exponential Smoothing Method

Exponential smoothing is a weighted average method that assigns more weight to recent observations. It is particularly useful for detecting outliers in time series with trend or seasonal components.

- **Step 1 — Choose a smoothing factor $\alpha \in (0, 1]$**: A smaller $\alpha$ gives more weight to older observations (slower adaptation); a larger $\alpha$ gives more weight to recent observations (faster adaptation).

- **Step 2 — Compute the smoothed values**: The smoothed value at time $t$ is:
$$\begin{cases} s_0 = y_0 \\ \dots \\s_t = \alpha\, y_t + (1 - \alpha)\, s_{t-1} \end{cases}$$
where $y_t$ is the observed value and $s_{t-1}$ is the previous smoothed value.

- **Step 3 — Compute the deviation**: $\delta_t = y_t - s_t$

- **Step 4 — Set a threshold**: Compute the standard deviation of all deviations $\{\delta_t\}$ and flag points where $|\delta_t|$ exceeds a multiple (e.g. 3×) of that standard deviation.

- **Step 5 — Visualise the results**: Plot the original series, the smoothed values, and any flagged outliers.

In [ ]:
def exponential_smoothing_anomaly_detector(df, alpha=0.5, threshold=3):
    # make of the copy of the data as the function will modify it
    df_es = df.copy()

    # step 2 - calculate the smoothed values
    df_es['s'] = df_es.kw
    for t_minus_1, t in zip(df_es.index[:-1], df_es.index[1:]):
        df_es.loc[t, 's'] = alpha * df_es.loc[t, 'kw'] + (1-alpha)*df_es.loc[t_minus_1, 's']

    # step 3 - calculate the deviation
    df_es['deviation'] = df_es.kw - df_es.s

    # step 4 - calculate the standard deviation of the deviations
    std_dev = df_es.deviation.std()

    # Identify anomalies - those points whose absolute deviation exceeds threshold x standard deviations
    query = (df_es.deviation < -threshold*std_dev) | (df_es.deviation > threshold*std_dev)
    potential_outliers = df_es.kw[query]

    print()
    print("Potential anomalies:")
    print(potential_outliers)
    return potential_outliers, df_es, std_dev

In [ ]:
# define the smoothing factor and threshold
alpha = 0.75
threshold = 2.5

# run exponential smoothing and identify potential outliers
potential_outliers, df_es, std_dev = exponential_smoothing_anomaly_detector(df, alpha=alpha, threshold=threshold)

In [ ]:
# Visualize the results
df_es.kw.plot(
    figsize=(15, 6),
    title='House consumption', 
    ylabel='Power (kW)', 
    xlabel='Date', 
    legend=False,
    grid=True,
    rot=45)
    
df_es.s.plot()

# plot the tolerance bandhe 
plt.scatter(
    x=potential_outliers.index.values, 
    y=potential_outliers.values, 
    color='red', 
    label='Anomalies',
    marker='X'
)

# plot the tolerance band
plt.fill_between(
    x=df_es.index.values,
    y1=(df_es.s.values-threshold*std_dev),
    y2=(df_es.s.values+threshold*std_dev),
    alpha=0.1, 
    color="g",
    label='Tolerance band'
)

plt.legend()

plt.show()

## Machine Learning Methods

Many applications require the ability to determine whether a new observation should be treated as an inlier (belonging to the same distribution as the training data) or as an outlier. Two important distinctions define the problem setting:

- **Outlier detection**: Outliers are defined as observations that deviate significantly from the norm *within the training data*. The model fits the high-density regions of the training data and flags deviations as anomalies. This is also known as **unsupervised anomaly detection**.

- **Novelty detection**: The training data is assumed to be clean (free of outliers). The goal is to detect whether a *new* observation is anomalous relative to what was seen during training. This is also known as **semi-supervised anomaly detection**.

In both cases, the estimators typically assume that anomalies occupy low-density regions of the feature space.

### Types of Available Data

The choice of anomaly detection technique depends on the availability and nature of labelled data:

- **Supervised**: The model is trained on fully labelled data (normal vs. anomalous). Requires a sufficient quantity of labelled examples of both classes, which is often impractical.

- **Semi-supervised**: Some data points are labelled; the remainder are not. The model learns from the partially labelled set. Some authors also use this term when training data is assumed to contain only normal samples.

- **Unsupervised**: No labels are used. The model learns the structure of the data entirely from the observations and flags deviations as anomalies.

- **Active semi-supervised**: The algorithm iteratively selects the most informative unlabelled samples and requests expert annotation. Useful when labelling is expensive and the task relies heavily on domain expertise.

### Methods

Several machine learning techniques can be used for outlier detection in time series data:

- **One-Class SVM**: Builds a model of normal behaviour in a high-dimensional feature space and identifies data points that fall outside the learned boundary.

- **Isolation Forest**: A tree-based algorithm that isolates anomalies by randomly partitioning the feature space. Anomalies require fewer splits to isolate and receive higher anomaly scores.

- **Autoencoder**: A neural network trained to reconstruct normal data. When presented with an anomalous input, the reconstruction error is higher than for normal data.

- **LSTM (Long Short-Term Memory)**: A recurrent neural network well-suited to sequential data. It learns the normal temporal patterns and flags significant deviations.

These methods can learn complex, high-dimensional patterns that are difficult to capture with traditional statistical techniques. The choice between them depends on the characteristics of the data, the available labels, and the interpretability requirements of the application. The following sections demonstrate the first three.

#### One-Class SVM

One-Class SVM (Support Vector Machine) is an unsupervised anomaly detection method that does not require labelled anomalies for training. It models the normal behaviour of a dataset using a single class of observations and identifies data points that deviate from this learned boundary as outliers.

Unlike a standard SVM that separates two classes, One-Class SVM finds a hyperplane in a high-dimensional feature space that encloses the normal data with maximum margin. Points falling outside this boundary are classified as anomalies.

##### Hyperparameters

One-Class SVM has three key hyperparameters. Understanding their role is essential before using the model:

---

**`kernel`** — the similarity (kernel) function used to map data into a high-dimensional feature space where the separating hyperplane is found.

| Value | When to use |
|---|---|
| `'rbf'` *(default)* | Most cases — captures non-linear structure; best starting point for time series |
| `'linear'` | High-dimensional data with many features |
| `'poly'` | When polynomial relationships are expected |

---

**`nu` ($\nu \in (0, 1]$)** — the most interpretable parameter. It simultaneously acts as:
- an **upper bound** on the fraction of training points classified as outliers, and
- a **lower bound** on the fraction of support vectors.

In practice, set `nu` to the **expected contamination rate** of your data:

-  Expect ~1% outliers → nu = 0.01
- Expect ~5% outliers → nu = 0.05

Then:

- Higher `nu` → looser boundary → more points flagged as outliers
- Lower `nu` → tighter boundary → fewer points flagged as outliers

---

**`gamma`** — controls the width of the RBF kernel (only relevant when `kernel='rbf'`):

$$K(x, x') = \exp\!\left(-\gamma \|x - x'\|^2\right)$$

| Value | Effect |
|---|---|
| High | Narrow influence per point → tight, jagged boundary → risk of overfitting |
| Low | Wide influence per point → smooth, broad boundary → risk of underfitting |
| `'scale'` *(default)* | $\gamma = 1 / (n\_\text{features} \times \text{Var}(X))$ — good general default |
| `'auto'` | $\gamma = 1 / n\_\text{features}$ |

---

##### Tuning strategy

Because One-Class SVM is **unsupervised** (no class labels), standard cross-validation cannot be applied directly. Three practical approaches are:

1. **Domain knowledge** — if the expected outlier fraction is known, set `nu` to that value directly.

2. **Visual inspection** — train with a range of `nu` values and plot the flagged points on the time series; choose the value that matches domain expectations.

3. **Decision function score** — plot `clf.decision_function(X)` over time; the score is positive for inliers and negative for outliers, and its magnitude indicates confidence.

> **Recommended order:** fix `kernel='rbf'` and `gamma='scale'`, tune `nu` first (it has the most direct interpretation), then refine `gamma` if the boundary appears too tight or too loose.

In [ ]:
import numpy as np
from sklearn.svm import OneClassSVM

# load data and resample it to hourly consumption
df_hourly = pd.read_csv('./data/house_consumption_TS/house_consumption.csv', parse_dates=['date'])\
    .set_index('date')\
    .resample('h')\
    .mean()\
    .fillna(0)

# select the data for the months of September to December 2022
mask = (df_hourly.index >= '2022-09-01') & (df_hourly.index <= '2022-12-31')
df_hourly = df_hourly[mask]


# build a dataframe with the consumption and associated hour of the day - adding features helps the method to identify anomalies, e.g., a anomalous consumption at a certain time
data = pd.DataFrame()
data.index = df_hourly.index
data['kw'] = df_hourly.kw
data['h'] = df_hourly.index.hour

# other possible features could be the week day, month, etc
# data['wd'] = df_hourly.index.weekday
# data['month'] = df_hourly.index.month

data.head()

##### Example 1 - only the consumption and the hour of the day

In this example, the input data is a dataframe with two columns: consumption and hour of the day. This does not capture the temporal patterns of the data, but it is a good starting point to understand the behavior of the model.

In [ ]:
# Train the One-Class SVM model
clf = OneClassSVM(
    nu=0.005,    
    kernel="rbf",
    gamma='scale'
)

clf.fit(data)

# Signed distance is positive for an inlier and negative for an outlier. The decision function returns the signed distance to the hyperplane.
scores = clf.decision_function(data)

# find the samples outside the threshold
query = scores < 0
potential_outliers = data[query]

potential_outliers.head()

# Visualize the results
ax = data.kw.plot(figsize=(28,8))
ax.scatter(x=potential_outliers.index.values, y=data.kw[potential_outliers.index], marker='X', c='r', s=100)

##### Example 2 - a window of consecutive hourly values

In this case, the input data is a dataframe with a window of consecutive hourly values. This captures the temporal patterns of the data, allowing the model to identify anomalies that occur at certain times of the day.

To prepare the data for the One-Class SVM model, we can create a sliding window of size 24 (hours). This "captures" the behaviour of the consumption at a certain moment. The window size is a hyperparameter that can be tuned to improve the performance of the model.

In [ ]:
# Prepare the data by creating a sliding window of size 24 (hours).
# This "captures" the behaviour of the consumption at a certain moment
window_size = 24

# create a sliding window of size 24 (hours)
X = np.array([data.iloc[i:i+window_size, 0] for i in range(len(data)-window_size+1)])
print('Shape of the data after creating the sliding window:', X.shape)

# flatten the data to a 2D array
X = np.reshape(X, (X.shape[0], -1))
print('Shape of the data after flattening:', X.shape)

# convert to a dataframe
X = pd.DataFrame(X)

# to use the index of the first / last reading as reference - uncomment one of the following lines
# X.index = data.index[:-(window_size-1)] # use the index of the first reading as reference
X.index = data.index[window_size-1:] # use the index of the last reading as reference

X.head()

In [ ]:
# Train the One-Class SVM model
clf = OneClassSVM(
    nu=0.005,    
    kernel="rbf",
    gamma='scale'
)

clf.fit(X)

# Signed distance is positive for an inlier and negative for an outlier. The decision function returns the signed distance to the hyperplane.
scores = clf.decision_function(X)
pd.Series(scores).plot(
    figsize=(28,8),
    title='One-Class SVM scores',
    ylabel='Signed distance to the hyperplane',
    xlabel='Time',
    legend=False,
    grid=True,
    rot=45
)

The model is trained with `nu=0.005` (expecting ~0.5 % of windows to be anomalous), `kernel='rbf'` (non-linear boundary), and `gamma='scale'` (width auto-scaled to the data variance). The `decision_function` returns a signed distance: **positive → inlier**, **negative → outlier**.

In [ ]:
# find the samples outside the threshold
query = scores < 0
potential_outliers = X[query]

potential_outliers.head()

In [ ]:
# Visualize the results
ax = data.kw.plot(figsize=(28,8))
ax.scatter(x=potential_outliers.index.values, y=data.kw[potential_outliers.index], marker='X', c='r', s=100)

One of the main problems with the **explainability** of the One-Class SVM is that it is a **black box** model, which means that it is difficult to understand how the model arrived at its predictions. This lack of transparency can make it challenging to interpret the results and understand the underlying reasoning behind them.

One of the reasons for this lack of transparency is that the One-Class SVM is a non-parametric model, which means that it does not have a fixed number of parameters that can be easily interpreted. Instead, the model is trained to optimize a complex objective function, which makes it difficult to understand how the model works internally.

Another reason for the lack of interpretability is that the One-Class SVM uses a kernel function to map the input data to a higher-dimensional space, where it can be more easily separated by a hyperplane. The choice of kernel function can have a significant impact on the performance of the model, but it can be difficult to understand how the kernel function affects the model's behavior.

To address these challenges, researchers are exploring different methods for improving the explainability of the One-Class SVM, such as using interpretable kernel functions or post-hoc explanations that provide insights into the model's decision-making process. However, achieving high levels of interpretability while maintaining the accuracy and effectiveness of the One-Class SVM remains an ongoing research challenge.


#### Isolation Forest

Isolation Forest is an ensemble anomaly detection algorithm that identifies outliers by exploiting the fact that anomalies are few and structurally different from normal observations. It works by constructing a tree structure where each split is made by randomly selecting a feature and a value within the range of the feature, which isolates the observations by creating partitions or sub-trees.

The basic idea behind Isolation Forest is that **anomalies are more likely to be isolated or separated from the rest of the data points in a few number of splits**, as opposed to normal data points which are more likely to be grouped together. Thus, the algorithm isolates anomalies faster and with fewer splits, compared to normal data points.

In the isolation process, the algorithm randomly selects a feature and a value, and creates a partition that isolates some of the data points. It repeats this process recursively until all data points are isolated, or until it reaches a predefined number of splits. The depth of each tree is limited, so as to avoid overfitting to the training data.

After constructing the trees, the algorithm assigns each data point an anomaly score based on the **average path length** needed to isolate it across all trees. Short paths indicate anomalies (few splits needed to isolate); long paths indicate normal points (harder to isolate). The anomaly score is:

$$s(x, n) = 2^{-\dfrac{E(h(x))}{c(n)}}$$

where:
- $x$ is the data point being scored
- $n$ is the **number of training samples**
- $h(x)$ is the **path length** of $x$ in a single isolation tree (number of edges from the root to the node where $x$ is isolated)
- $E(h(x))$ is the **expected path length** of $x$, averaged across all trees in the ensemble
- $c(n)$ is a **normalisation constant** — the average path length of an unsuccessful binary search tree search over $n$ nodes:

$$c(n) = 2H(n-1) - \frac{2(n-1)}{n}, \qquad H(i) = \ln(i) + 0.5772\ldots \text{ (Euler–Mascheroni constant)}$$

Interpretation of the score:
- $s \to 1$ — very short path → point is easy to isolate → **anomaly**
- $s \approx 0.5$ — path ≈ average → **normal point**
- $s \to 0$ — very long path → point blends in with the bulk of the data → **definitely normal**

Isolation Forest has several advantages over other anomaly detection methods, such as the ability to handle high-dimensional datasets and large datasets, and the ability to detect anomalies in a short amount of time. However, it may not perform well in detecting anomalies that are closely interleaved with normal data, and may require careful tuning of its hyperparameters to achieve good performance.

![./images/IF_anomalous_point.png](./images/IF_anomalous_point.png)

##### Example 1 - raw time series values (consumption, hour of the day)

The following example demonstrates the Isolation Forest implementation from scikit-learn applied directly to the raw time series values (without a sliding window).

In [ ]:
data.head()

In [ ]:
from sklearn.ensemble import IsolationForest

clf = IsolationForest(n_estimators=100, # fit 50 trees
                      contamination=0.01) # move this value to be more/less strict

predictions = clf.fit_predict(data)

potencial_outliers = data[predictions == -1]
potencial_outliers

In [ ]:
# Visualize the results
ax = data.kw.plot(figsize=(28,8))
ax.scatter(x=potencial_outliers.index.values, y=potencial_outliers.kw, marker='X', c='r', s=100)

##### Example 2 - a window of consecutive hourly values

The following example uses the same sliding-window strategy as the One-Class SVM above. Each window of `window_size` consecutive hourly values is treated as a single feature vector, allowing the algorithm to capture temporal patterns rather than individual point values.

In [ ]:
window_size = 12

X = np.array([data.iloc[i:i+window_size, 0] for i in range(len(data)-window_size+1)])
X = np.reshape(X, (X.shape[0], -1))
X = pd.DataFrame(X)

# to use the index of the last reading as reference
X.index = data.index[window_size-1:]

X.head()

In [ ]:
clf = IsolationForest(n_estimators=50,
                      contamination=0.01)
predictions = clf.fit_predict(X)  # fit 10 trees  

potencial_outliers = X[predictions == -1]
potencial_outliers

In [ ]:
# Visualize the results
import matplotlib.pyplot as plt
ax = data.kw.plot(figsize=(28,8))
ax.scatter(x=potential_outliers.index.values, y=data.kw[potential_outliers.index], marker='X', c='r', s=100)

The Isolation Forest algorithm is generally **considered more interpretable than some other anomaly detection methods**, such as neural networks, because it is based on a simple and intuitive concept of isolating anomalies from the rest of the data. However, the algorithm is still not as transparent or easily explainable as some other machine learning algorithms, such as linear regression or decision trees.

One of the challenges with explaining Isolation Forest is that it is an ensemble method, meaning that it combines the results of multiple trees to generate a final anomaly score for each data point. The contribution of each tree to the final score may not be immediately obvious, making it difficult to understand how the algorithm arrived at its decision.

To address this challenge, researchers have proposed various techniques for visualizing and interpreting Isolation Forest models. For example, one approach is to plot the feature importance values, which show how much each feature contributes to the isolation of anomalies. Another approach is to visualize the paths taken by the algorithm to isolate anomalies, which can provide insights into the structure of the data and the nature of the anomalies.

In general, while Isolation Forest is not a completely transparent or explainable algorithm, it is still possible to gain some insights into how it works and how it arrived at its decision through various visualization and interpretation techniques.

#### Autoencoder

Autoencoders are a type of neural network that can be used for anomaly detection. The basic idea is to **train the autoencoder on a dataset of normal data points and then use it to reconstruct new data points. When the autoencoder is presented with an anomaly, it may not be able to reconstruct it accurately, indicating that it is an outlier or anomaly**.

The anomaly detection process using autoencoders typically involves two steps: training and testing. During training, the autoencoder is trained on a dataset of normal data points, which are passed through the encoder to produce a lower-dimensional representation, and then passed through the decoder to reconstruct the original data. The objective is to minimize the reconstruction error between the input and the output, which encourages the autoencoder to learn a compressed representation of the normal data.

During testing, the autoencoder is presented with new data points, and the reconstruction error is calculated between the input and the output. **If the reconstruction error is above a certain threshold, the data point is classified as an anomaly.**

The **interpretability of autoencoders can be challenging**, as they are often used in a black-box manner, and it may be difficult to understand the underlying factors that contribute to the reconstruction error or the decision to classify a data point as an anomaly. However, techniques such as visualization of the encoded representations or feature importance analysis can provide some insights into the behavior of the autoencoder.

![./images/autoencoder.png](./images/autoencoder.png)

In [ ]:
window_size = 24

X = np.array([data.iloc[i:i+window_size, 0] for i in range(len(data)-window_size+1)])
X = np.reshape(X, (X.shape[0], -1))
X = pd.DataFrame(X)

# to use the index of the last reading as reference or the first reading as reference - uncomment one of the following lines
# X.index = data.index[window_size-1:]
X.index = data.index[:-(window_size-1)]

# define the percentage of data to use
n_train = int(len(X)*0.75)

# define the training and test sets
X_train = X.iloc[:n_train]
X_test = X.iloc[n_train:, :]

print(X_train.shape)
print(X_test.shape)

In [ ]:
from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout

class AutoEncoder(Model):
    def __init__(self, input_dim, code_size=8):
        super().__init__()

        self.encoder = Sequential([
            Dense(32, activation='relu', input_shape=(input_dim, )),
            # Dense(16, activation='relu'),
            # Dense(8, activation='relu'),
            Dense(code_size, activation='relu'),
        ])
        self.decoder = Sequential([
            # Dense(8, activation='relu'),
            # Dense(16, activation='relu'),
            Dense(32, activation='relu'),
            Dense(input_dim, activation='linear')
        ])

    def call(self, inputs):
        encoded = self.encoder(inputs)
        decoded = self.decoder(encoded)
        return decoded
  
# configurations of model
model = AutoEncoder(
    input_dim=X_train.shape[1], 
    code_size=12
)

model.compile(
    loss='mse', 
    metrics=['mse'], 
    optimizer='adam'
)

history = model.fit(
    X_train,
    X_train,
    verbose=1,
    epochs=50,
    batch_size=64,
    validation_data=(X_test, X_test)
)

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.xlabel('Epochs')
plt.ylabel('MSLE Loss')
plt.legend(['loss', 'val_loss'])
plt.show()

The loss curve suggests that ~10 epochs are sufficient for convergence. The model is trained for 12 epochs.

In [ ]:
history = model.fit(
    X_train,
    X_train,
    verbose=1,
    epochs=20,
    batch_size=64,
    validation_data=(X_test, X_test)
)

The threshold is calculated as the mean of the reconstruction errors plus one standard deviation. This is a common technique for setting thresholds in anomaly detection algorithms.

In [ ]:
import tensorflow as tf

def find_threshold(model, X, k=3):
  reconstructions = model.predict(X)

  # provides losses of individual instances
  reconstruction_errors = tf.keras.losses.mse(reconstructions, X)

  # threshold for anomaly scores
  threshold = np.mean(reconstruction_errors.numpy()) + k*np.std(reconstruction_errors.numpy())

  return threshold

threshold = find_threshold(model, X_train, k=5)
print(f"Threshold: {threshold}")

In [ ]:
def get_predictions(model, X_test, threshold):
  predictions = model.predict(X_test)

  # provides losses of individual instances
  errors = tf.keras.losses.mse(predictions, X_test)

  # 0 = anomaly, 1 = normal
  anomaly_mask = pd.Series(errors) > threshold

  #preds = anomaly_mask.map(lambda x: False if x == True else 1.0)
  return anomaly_mask


predictions = get_predictions(model, X_test, threshold)

potential_anomalies = X_test[predictions.values]
print(f'Number of anomalies: {len(potential_anomalies)}')
potential_anomalies

In [ ]:
# Visualize the results over the test set
ax = X_test.iloc[:,0].plot(
    figsize=(20,5), 
    alpha=0.75,
    label='Normal',
    title='Anomalies in the data',
    grid=True,
    rot=45
)

_ = ax.scatter(
    x=potential_anomalies.index.values, 
    y=X_test.loc[potential_anomalies.index, 0], 
    c='r', 
    label='Anomalies',
    marker='X'
)
plt.show()

Plorting the real data vs the predicted data

In [ ]:
# Visualize the results
ax = X_test.iloc[:,0].plot(figsize=(20,5), alpha=0.75)


predicted = pd.Series(model.predict(X_test)[:, 0])
predicted.index = X_test.index
pd.Series(predicted).plot(ax=ax, c='r')

plt.show()

Interpretability is limited, but examining the reconstruction error over time can reveal periods where the model struggled to reproduce the input — often corresponding to changes in local consumption behaviour that may indicate anomalies.

### Ensembles

Anomaly detection ensembles are a collection of multiple anomaly detection models that work together to identify anomalies in data. Ensembles are commonly used in anomaly detection because they can improve the overall accuracy and robustness of the detection process.

There are several types of anomaly detection ensembles, including:

- **Voting ensembles**: In a voting ensemble, each model in the ensemble independently analyzes the data and casts a vote for whether it is anomalous or not. The final decision is made based on the majority vote of the models.

- **Stacking ensembles**: In a stacking ensemble, the output of each model in the ensemble is used as input to a higher-level model, which combines their outputs to make a final decision.

- **Boosting ensembles**: In a boosting ensemble, models are trained sequentially, with each model trying to correct the errors made by the previous models.

- **Bagging ensembles**: In a bagging ensemble, multiple models are trained independently on different subsets of the data, and their outputs are combined to make a final decision.

Ensembles can be effective in anomaly detection because they can help reduce the risk of false positives or false negatives that can occur with single models. By combining the outputs of multiple models, ensembles can also help improve the overall accuracy of anomaly detection.
